# ML Model Pipelines 

Clean, leakage-safe modeling notebook with separate linear vs non-linear workflows.


## Goals

- Preserve ordinal vs nominal categorical handling.
- Linear models (`LinearRegression`, `Lasso`, `Ridge`): apply log transform and IQR outlier removal.
- Non-linear models (`XGBoost`, `RandomForest`): keep outliers and do not apply log transforms.
- Split train/validation early.
- Run GridSearchCV on training split only.
- Validate once on holdout validation split.
- Retrain best configs on full train data and predict on test data.


In [2]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")


In [3]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data" / "raw").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "raw"
REPORTS_DIR = BASE_DIR / "reports" / "submissions"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data.preprocess import clean_test_data, clean_train_data
from features.features import add_engineered_features, drop_highly_correlated_features


In [4]:
train_df_raw = pd.read_csv(DATA_DIR / "train.csv")
test_df_raw = pd.read_csv(DATA_DIR / "test.csv")

train_df = add_engineered_features(clean_train_data(train_df_raw))
test_df = add_engineered_features(clean_test_data(test_df_raw, train_df_raw))

train_df = drop_highly_correlated_features(train_df)
test_df = drop_highly_correlated_features(test_df)

TARGET_COL = "SalePrice"
X = train_df.drop(columns=[TARGET_COL]).copy()
y = train_df[TARGET_COL].copy()
X_test = test_df.copy()

X.shape, y.shape, X_test.shape


((1460, 82), (1460,), (1459, 82))

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_val.shape


((1168, 82), (292, 82))

In [6]:
ordinal_categories = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "PoolQC": ["Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "Fence": ["MnWw", "GdWo", "MnPrv", "GdPrv"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "PavedDrive": ["N", "P", "Y"],
}

categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
ordinal_cols = [c for c in categorical_cols if c in ordinal_categories]
nominal_cols = [c for c in categorical_cols if c not in ordinal_cols]
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

len(numeric_cols), len(ordinal_cols), len(nominal_cols)


(39, 20, 23)

In [7]:
class SelectiveLogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to selected numeric columns only."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns and pd.api.types.is_numeric_dtype(X[col]):
                min_val = X[col].min()
                if pd.notna(min_val) and min_val <= -1:
                    X[col] = X[col] + abs(min_val) + 1.0
                X[col] = np.log1p(X[col])
        return X


def remove_outliers_iqr_low_rate_columns(X_df, y_series, threshold_pct=10.0):
    """Drop rows that are outliers in columns whose outlier rate is < threshold_pct."""
    X_df = X_df.copy()
    y_series = y_series.copy()

    outlier_rates = []
    numeric = X_df.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric:
        q1 = X_df[col].quantile(0.25)
        q3 = X_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (X_df[col] < lower) | (X_df[col] > upper)
        count = int(mask.sum())
        if count > 0:
            outlier_rates.append((col, 100 * count / len(X_df)))

    drop_candidate_cols = [col for col, pct in outlier_rates if pct < threshold_pct]

    rows_to_drop = pd.Series(False, index=X_df.index)
    for col in drop_candidate_cols:
        q1 = X_df[col].quantile(0.25)
        q3 = X_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        rows_to_drop |= (X_df[col] < lower) | (X_df[col] > upper)

    X_clean = X_df.loc[~rows_to_drop].copy()
    y_clean = y_series.loc[~rows_to_drop].copy()
    return X_clean, y_clean, drop_candidate_cols, int(rows_to_drop.sum())


In [8]:
def build_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    ordinal_categories_in_order = [ordinal_categories[col] for col in ordinal_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_cols),
            (
                "ord",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OrdinalEncoder(
                            categories=ordinal_categories_in_order,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    ),
                ]),
                ordinal_cols,
            ),
            (
                "nom",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                nominal_cols,
            ),
        ],
        remainder="drop",
    )
    return preprocessor


def evaluate_regression(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


## Linear Models (Log + Outlier Removal)


In [9]:
X_train_linear, y_train_linear, linear_drop_cols, dropped_rows = remove_outliers_iqr_low_rate_columns(
    X_train,
    y_train,
    threshold_pct=10.0,
)

print(f"Linear outlier columns used for row drop: {len(linear_drop_cols)}")
print(f"Rows dropped from training split: {dropped_rows}")
print(f"Train shape before: {X_train.shape} | after: {X_train_linear.shape}")


Linear outlier columns used for row drop: 24
Rows dropped from training split: 466
Train shape before: (1168, 82) | after: (702, 82)


In [10]:
linear_log_columns = [
    col for col in ["OverallQual", "Age", "TotalSF", "TotalBath", "LotArea", "MasVnrArea"]
    if col in X_train.columns
]

linear_base_pipeline = Pipeline([
    ("log_transform", SelectiveLogTransformer(columns=linear_log_columns)),
    ("preprocessor", build_preprocessor(scale_numeric=True)),
    ("model", LinearRegression()),
])

linear_model_grids = {
    "linear_regression": {
        "pipeline": linear_base_pipeline,
        "param_grid": {
            "model": [LinearRegression()],
        },
    },
    "lasso": {
        "pipeline": linear_base_pipeline,
        "param_grid": {
            "model": [Lasso(max_iter=20000, random_state=42)],
            "model__alpha": [0.0005, 0.001, 0.005, 0.01, 0.05],
        },
    },
    "ridge": {
        "pipeline": linear_base_pipeline,
        "param_grid": {
            "model": [Ridge(random_state=42)],
            "model__alpha": [0.1, 0.5, 1.0, 5.0, 10.0],
        },
    },
}


In [11]:
results = []
best_estimators = {}

for model_name, cfg in linear_model_grids.items():
    gs = GridSearchCV(
        estimator=cfg["pipeline"],
        param_grid=cfg["param_grid"],
        scoring="neg_root_mean_squared_error",
        cv=5,
        n_jobs=-1,
    )
    gs.fit(X_train_linear, y_train_linear)

    val_pred = gs.best_estimator_.predict(X_val)
    metrics = evaluate_regression(y_val, val_pred)
    metrics.update(
        {
            "model": model_name,
            "family": "linear",
            "best_params": gs.best_params_,
        }
    )
    results.append(metrics)
    best_estimators[model_name] = gs.best_estimator_

pd.DataFrame(results).sort_values("rmse")


,rmse,mae,r2,model,family,best_params
1,39202.768255,24419.113817,0.799636,lasso,linear,"{'model': Lasso(max_iter=20000, random_state=4..."
0,39705.609527,24626.362830,0.794463,linear_regression,linear,{'model': LinearRegression()}
2,40292.886843,23823.345678,0.788338,ridge,linear,"{'model': Ridge(random_state=42), 'model__alph..."


In [24]:
pd.DataFrame(results)['best_params'][1]

{'model': Lasso(max_iter=20000, random_state=42), 'model__alpha': 0.05}

In [26]:
pd.DataFrame(results)['best_params'][2]

{'model': Ridge(random_state=42), 'model__alpha': 5.0}

## Non-Linear Models (No Log, Keep Outliers)


In [12]:
nonlinear_base_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(scale_numeric=False)),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

nonlinear_model_grids = {
    "random_forest": {
        "pipeline": nonlinear_base_pipeline,
        "param_grid": {
            "model": [RandomForestRegressor(random_state=42, n_jobs=-1)],
            "model__n_estimators": [300, 600],
            "model__max_depth": [None, 12, 20],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
        },
    }
}

try:
    from xgboost import XGBRegressor

    nonlinear_model_grids["xgboost"] = {
        "pipeline": Pipeline([
            ("preprocessor", build_preprocessor(scale_numeric=False)),
            (
                "model",
                XGBRegressor(
                    objective="reg:squarederror",
                    random_state=42,
                    n_estimators=500,
                    n_jobs=-1,
                ),
            ),
        ]),
        "param_grid": {
            "model__n_estimators": [400, 700],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    }
    print("XGBoost is available and included.")
except Exception:
    print("XGBoost not available in environment. Running RandomForest only.")


XGBoost is available and included.


In [13]:
for model_name, cfg in nonlinear_model_grids.items():
    gs = GridSearchCV(
        estimator=cfg["pipeline"],
        param_grid=cfg["param_grid"],
        scoring="neg_root_mean_squared_error",
        cv=5,
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    val_pred = gs.best_estimator_.predict(X_val)
    metrics = evaluate_regression(y_val, val_pred)
    metrics.update(
        {
            "model": model_name,
            "family": "non_linear",
            "best_params": gs.best_params_,
        }
    )
    results.append(metrics)
    best_estimators[model_name] = gs.best_estimator_

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df


,rmse,mae,r2,model,family,best_params
0,24669.584836,15076.664062,0.920657,xgboost,non_linear,"{'model__colsample_bytree': 0.8, 'model__learn..."
1,29538.810609,17386.769657,0.886245,random_forest,non_linear,"{'model': RandomForestRegressor(n_jobs=-1, ran..."
2,39202.768255,24419.113817,0.799636,lasso,linear,"{'model': Lasso(max_iter=20000, random_state=4..."
3,39705.609527,24626.362830,0.794463,linear_regression,linear,{'model': LinearRegression()}
4,40292.886843,23823.345678,0.788338,ridge,linear,"{'model': Ridge(random_state=42), 'model__alph..."


## Retrain on Full Data and Save Submissions


In [14]:
submission_paths = {}

X_full_linear, y_full_linear, _, _ = remove_outliers_iqr_low_rate_columns(
    X,
    y,
    threshold_pct=10.0,
)

for row in results_df.itertuples(index=False):
    model_name = row.model
    estimator = best_estimators[model_name]

    if row.family == "linear":
        estimator.fit(X_full_linear, y_full_linear)
    else:
        estimator.fit(X, y)

    test_pred = estimator.predict(X_test)
    test_pred = np.maximum(test_pred, 0)

    submission = pd.DataFrame({"Id": test_df_raw["Id"], "SalePrice": test_pred})

    out_path = REPORTS_DIR / f"{model_name}_refactored.csv"
    submission.to_csv(out_path, index=False)
    submission_paths[model_name] = str(out_path)

submission_paths


{'xgboost': '/home/amraas/projects/realestatecons/reports/submissions/xgboost_refactored.csv',
 'random_forest': '/home/amraas/projects/realestatecons/reports/submissions/random_forest_refactored.csv',
 'lasso': '/home/amraas/projects/realestatecons/reports/submissions/lasso_refactored.csv',
 'linear_regression': '/home/amraas/projects/realestatecons/reports/submissions/linear_regression_refactored.csv',
 'ridge': '/home/amraas/projects/realestatecons/reports/submissions/ridge_refactored.csv'}

In [15]:
results_df


,rmse,mae,r2,model,family,best_params
0,24669.584836,15076.664062,0.920657,xgboost,non_linear,"{'model__colsample_bytree': 0.8, 'model__learn..."
1,29538.810609,17386.769657,0.886245,random_forest,non_linear,"{'model': RandomForestRegressor(n_jobs=-1, ran..."
2,39202.768255,24419.113817,0.799636,lasso,linear,"{'model': Lasso(max_iter=20000, random_state=4..."
3,39705.609527,24626.362830,0.794463,linear_regression,linear,{'model': LinearRegression()}
4,40292.886843,23823.345678,0.788338,ridge,linear,"{'model': Ridge(random_state=42), 'model__alph..."
